# 02. Binary Reversing Lecture Lab

## 강의 목표

작은 C 프로그램을 직접 컴파일하고, binary에서 다음을 찾는다.

- 파일 포맷/아키텍처
- 문자열 단서
- `main` 함수
- `strcmp` 호출
- branch 조건
- 동적 디버깅에서 register/memory 확인

이 노트북은 pwn/exploit이 아니라 **reversing workflow**를 익히기 위한 강의자료다.

## 0. Setup

macOS 기준으로 진행한다.

사용 도구: `clang`, `file`, `strings`, `otool`, `lldb`.
노트북에서는 `subprocess`로 static analysis 명령을 실행한다.

In [ ]:
from pathlib import Path
import subprocess, textwrap, os, json, re

ROOT = Path('/Users/kbsoo/Codes/projects/interp')
LAB = ROOT / 'reversing' / 'labs'
LAB.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=LAB):
    print('$', ' '.join(map(str, cmd)))
    p = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    print('exit:', p.returncode)
    return p

## 1. Source Code

소스에서는 `strcmp(argv[1], "secret")`가 명확하다.
Binary에서는 이 정보가 문자열/함수 호출/branch로 흩어진다.
Reversing은 이 흩어진 정보를 다시 로직으로 복원하는 일이다.

In [ ]:
source = """
#include <stdio.h>
#include <string.h>

int main(int argc, char **argv) {
    if (argc != 2) return 1;

    if (strcmp(argv[1], "secret") == 0) {
        puts("correct");
    } else {
        puts("wrong");
    }

    return 0;
}
"""
path = LAB / 'check.c'
path.write_text(source)
print(path)
print(source)

## 2. Compile O0 / O2

- `-O0 -g`: 초보 분석용. 원본 구조가 많이 남아있다.
- `-O2`: 최적화 때문에 구조가 바뀔 수 있다.

In [ ]:
run(['clang', '-O0', '-g', 'check.c', '-o', 'check_O0'])
run(['clang', '-O2', 'check.c', '-o', 'check_O2'])

## 3. Run Program

입력에 따라 `correct`/`wrong`이 갈린다.

In [ ]:
run(['./check_O0', 'secret'])
run(['./check_O0', 'nope'])

## 4. Static Analysis 1 — file / strings

처음 보는 binary는 거의 항상 이 둘부터 본다.

In [ ]:
run(['file', 'check_O0'])
run(['strings', 'check_O0'])

### Exercise 1

위 `strings` 결과에서 아래를 찾아 적어라.

- correct/wrong 문자열이 보이는가?
- secret 문자열이 보이는가?
- 이걸로 이미 정답을 알 수 있는가?
- 그렇다면 이 문제는 왜 쉬운가?

## 5. Static Analysis 2 — Symbols / Disassembly

macOS Mach-O에서는 `otool`을 쓴다. Linux ELF면 `objdump -d -M intel`을 쓰면 된다.
아래 출력에서 `main`, `_strcmp`, `_puts` 근처를 찾는다.

In [ ]:
run(['nm', 'check_O0'])

In [ ]:
p = subprocess.run(['otool', '-tV', 'check_O0'], cwd=LAB, text=True, capture_output=True)
text = p.stdout
print(text[:5000])
print('... truncated ... total chars:', len(text))

### Exercise 2

Disassembly에서 다음을 찾아라.

- `main` 시작 주소
- `argc != 2` 체크로 보이는 compare/branch
- `strcmp` 호출
- `strcmp` 반환값을 검사하는 branch
- `puts("correct")`와 `puts("wrong")`로 갈라지는 지점

처음에는 assembly를 한 줄씩 완벽히 해석하려 하지 말고, **call과 branch를 먼저 표시**해라.

## 6. Dynamic Analysis — lldb로 직접 확인

노트북에서 lldb를 완전 interactive하게 쓰기는 불편하므로, 이 부분은 터미널에서 직접 한다.

```bash
cd /Users/kbsoo/Codes/projects/interp/reversing/labs
lldb ./check_O0
```

추천 명령:

```lldb
(lldb) breakpoint set --name main
(lldb) run test
(lldb) disassemble --frame
(lldb) register read
(lldb) thread step-inst
(lldb) continue
```

볼 것:

- `argv[1]`가 어디에 있는가?
- `strcmp` 호출 전 인자가 어떻게 전달되는가?
- `strcmp` 이후 반환값이 어떤 register에 있는가?
- branch가 correct/wrong 중 어디로 가는가?

## 7. Writeup Skeleton

아래 내용을 `reversing/writeups/001-simple-string-check.md`에 채운다.

```md
# 001. Simple String Check

## Static observations
- Format:
- Architecture:
- Interesting strings:
- Imports:
- Candidate function:

## Dynamic observations
- Breakpoint:
- argv[1] location:
- strcmp call:
- branch condition:

## Recovered pseudo-code

if (argc != 2) return 1;
if (strcmp(argv[1], "secret") == 0) puts("correct");
else puts("wrong");

## What I learned
- 
```

## 8. Exercise 3 — 직접 변형하기

`secret`을 그대로 두면 `strings`로 답이 너무 쉽게 보인다. 다음 단계는 XOR crackme다.

과제:

- [ ] `strings`로 정답이 안 보이게 만들기
- [ ] Ghidra/otool/lldb로 XOR 로직 찾기
- [ ] Python solver 작성
- [ ] writeup 002 작성

In [ ]:
print('Next lab: implement XOR-based check and write solver')

## 9. Mech Interp와 연결되는 관점

이 lab에서 한 일:

- source-level intent가 binary에서 call/branch/data로 흩어진다.
- static analysis로 후보 로직을 세운다.
- dynamic analysis로 실제 실행 상태를 검증한다.

Mech interp에서도 비슷하다.

- behavior가 activation/head/layer에 흩어진다.
- attention/activation 관찰로 후보를 세운다.
- ablation/patching으로 causal하게 검증한다.